In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Part 1: Schema Validation

### Schema Validation in Delta Lake

Delta Lake is **schema-on-write** — before any data is written to a table, it's checked against the table's existing schema. If it doesn't match, the write is rejected. This is what prevents a data lake from turning into a pile of inconsistently-shaped files.

Here is the difference between schema-on-read vs schema-on-write:

<img src="https://github.com/afaqueahmad7117/databricks-masterclass/blob/main/delta_lake/docs/images/Schema%20On%20Read%20&%20Write.png?raw=true" height=500/>

## 1.1 Setup

In [33]:
source_df = (
    (
        spark.read.format('parquet')
        .load('abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/raw_data/invoices_1_100.parquet')
    )
    .select('customer_id', 'invoice_no', 'quantity', 'price')
    .withColumn(
        "customer_id", F.col("customer_id").cast("long"),
    )
    .withColumn(
        "quantity", F.col("quantity").cast("long"),
    )
    .withColumn(
        "price", F.col("price").cast("double"),
    )
)

In [35]:
source_df.printSchema()

In [34]:
display(source_df.limit(5))

In [53]:
source_delta_table = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/delta_lake_tutorial/schema_evolution/"
source_df.write.format('delta').mode('overwrite').option('overwriteSchema', True).save(source_delta_table)

source_delta_table_df = spark.read.format('delta').load(source_delta_table)

## Scenario 1: Column Order Validation

There are three separate things Delta checks. Each one can independently cause a write to fail:

#### 1. Column Names
Every column in the incoming data must exist in the target table with the **exact same name**.

```
Table schema:  customer_id, invoice_no, quantity, price
Data schema:   quantity, invoice_no, customer_id, price, invoice_date   ❌ extra column

AnalysisException: A schema mismatch detected when writing to the Delta table
```
Fix: drop/select only the columns that exist in the target, or rename to match.

#### 2. Column Types
Every column's type must either **match exactly** or be safely castable to the target's type. A common trap: reading a value from a source Parquet file often infers a wider type (`long`) than the target Delta column was defined with (`int`) — these don't auto-merge.

```
DELTA_FAILED_TO_MERGE_FIELDS: Failed to merge fields 'customer_id' and 'customer_id'
```
Fix: explicitly cast the offending column with `.withColumn("col", F.col("col").cast("target_type"))` before writing — don't rely on `mergeSchema` for this, since a genuine type conflict on an *existing* column isn't a schema evolution case, it's a straight mismatch.

#### 3. Column Position — **matching rules differ by write method**

This is the one to be careful about, since different PySpark write paths enforce this differently:

| Write method | Matches by | Notes |
|---|---|---|
| `.write.format("delta").mode("append").save(path)` | **Name** | Rejects extra/missing columns with a clear schema-mismatch error. Column *order* in your DataFrame doesn't matter. |
| `.write.insertInto("table_name")` | **Position** | Requires a catalog-registered table (not path-based). Ignores column names entirely — values are mapped purely by column order, which can silently insert data into the wrong columns if your DataFrame's column order doesn't match the table's. |
| `DeltaTable.merge(...)` | **Name** (explicit) | Matches by name; if source/target names differ, you must map them explicitly via `whenNotMatchedInsert(values={...})` — it won't guess. |

**Practical takeaway:** the everyday `.save()` / `.saveAsTable()` append path is name-based and reasonably safe — mismatched column order alone won't corrupt data, it'll just throw an error if names/types don't line up. The silent-corruption risk only shows up with `insertInto()`, since that's the one PySpark API that truly mirrors SQL's positional `INSERT INTO ... SELECT *` behavior.


In [49]:
new_data_df_v1 = spark.createDataFrame(
    [(105, "I178410", 501, 1500.0)],
    ['quantity', 'invoice_no', 'customer_id', 'price']
)
new_data_df_v1.show()

In [50]:
new_data_df_v1.printSchema()

In [51]:
new_data_df_v1.write.format('delta').mode('append').save(source_delta_table)

Using `Merge`

In [55]:
new_data_df_v2 = spark.createDataFrame(
    [(106, "I178410", 599, 1500.0)],
    ['quantity', 'invoice_no', 'customer_id', 'price']
)

new_data_df_v2.show()

In [56]:
target_table = DeltaTable.forPath(spark, source_delta_table)

(
    target_table.alias('t')
    .merge(
        new_data_df_v2.alias('s'),
        "t.customer_id == s.customer_id"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [57]:
source_delta_table_df.filter(F.col("customer_id")==599).show()

## Scenario 2: Data Type Validation

- Delta checks whether incoming values can be **cast** to the target column's type — not just whether they match exactly.
- A numeric string like `'99499'` succeeds (auto-castable to `INT`); a non-numeric string like `'ABC'` fails outright.
- In PySpark, `.write().save()` is generally **stricter upfront** than raw SQL `INSERT` — it won't silently attempt loose casts on write the way SQL's engine does, so you often need to cast explicitly (`.withColumn("col", F.col("col").cast("type"))`) before writing.
- Real-world trap: reading from Parquet frequently infers wider types (`long`, `double`) than your Delta table was defined with (`int`, `float`) — this is the most common type-mismatch source in practice, not deliberately "bad" data.


In [58]:
try:
    bad_type_df = spark.createDataFrame(
        [("ABC", "I45678", 10, 98.75, "2025-01-01")],
        ["customer_id", "invoice_number", "quantity", "price", "invoice_date"]
    )
    bad_type_df.write.format("delta").mode("append").save(source_delta_table)
except Exception as e:
    print("Failed as expected:", e)

In [64]:
ok_type_df = spark.createDataFrame(
    [("99499", "I45678", 10, 98.75)],
    ["customer_id", "invoice_no", "quantity", "price"]
).withColumn("customer_id", F.col("customer_id").cast("long"))  # explicit cast — see note below

ok_type_df.write.format("delta").mode("append").save(source_delta_table)

In [66]:
source_delta_table_df.filter(F.col('customer_id')==99499).show()

## Scenario 3 — Column Name Validation

- SQL's raw `INSERT INTO ... SELECT` matches by **position**, so it doesn't care if source column names differ from the target — this is what makes it dangerous (silent corruption).
- `MERGE` matches by **name** — if the source's column names don't match the target's, it fails with an unresolved-column error rather than guessing.
- In PySpark, `.write().save()` (append mode) also matches by **name** by default — closer to `MERGE` behavior than raw SQL `INSERT`, so a plain rename alone (with correct positions) will throw a schema-mismatch error instead of corrupting data.
- To reproduce SQL's true positional-matching risk in PySpark, you need `.write.insertInto("table_name")` on a catalog-registered table — that's the one API that ignores names and matches purely by column order.

In [ ]:
renamed_df = spark.createDataFrame(
    [(99999, "I12345", 10, 100.0)],
    ["c_id", "invoice_number", "qty", "price"]   # renamed columns
)

# Match target schema positionally by dropping to an RDD/tuple write, or renaming back:
renamed_df.write.format("delta").mode("append").save(source_delta_table_df)  # will fail: schema mismatch (names don't align)

## Scenario 4 — Nullability Validation

- Delta enforces column-level constraints (like `NOT NULL`) at **write time** — a row violating the constraint is rejected outright, not silently allowed in with a null.
- This isn't limited to `NOT NULL` — you can define other rules too (e.g., `price > 0`, `quantity >= 0`), and Delta checks all of them the same way on every write.
- Constraints are added via `ALTER TABLE ... ADD CONSTRAINT` (or defined at `CREATE TABLE` time) — there's no DataFrame-native equivalent, so this always goes through `spark.sql(...)` even in PySpark workflows.
- Practical effect: this catches bad data (missing required fields, invalid values) *before* it ever lands in the table, rather than relying on downstream validation or cleanup jobs to catch it later.


```python
from pyspark.sql import Row

null_row_df = spark.createDataFrame(
    [Row(customer_id=None, invoice_number=None, quantity=None, price=None, invoice_date=None)]
)

try:
    null_row_df.write.format("delta").mode("append").save(source_delta_table)
except Exception as e:
    print("Failed as expected — NOT NULL violated:", e)

# Fixed version — give customer_id an actual value
fixed_df = spark.createDataFrame(
    [(78912, None, None, None, None)],
    ["customer_id", "invoice_number", "quantity", "price", "invoice_date"]
)
fixed_df.write.format("delta").mode("append").save(source_delta_table)   # succeeds
```


In [69]:
extra_col_df = spark.createDataFrame(
    [(99999, "I12345", 10, 100.0, "2025-01-01", "VIP")],
    ["customer_id", "invoice_number", "quantity", "price", "invoice_date", "customer_type"]
)
try:
    extra_col_df.write.format("delta").mode("append").save(source_delta_table)
except Exception as e:
    print("Failed as expected — schema mismatch:", e)

In [95]:
merge_source_df = spark.range(176, 179).select(
    F.col("id").alias("customer_id")
).withColumn("invoice_no", F.lit("I999")) \
 .withColumn("quantity", F.lit(1)) \
 .withColumn("price", F.lit(50.0)) \
 .withColumn("invoice_date", F.lit("2025-01-01")) \
 .withColumn("customer_type", F.lit("VIP"))   # extra column not in target

merge_source_df =merge_source_df.withColumn('quantity', F.col('quantity').cast('long'))

merge_source_df.printSchema()

In [73]:
(
    target_table.alias("t")
    .merge(merge_source_df.alias("s"), "t.customer_id = s.customer_id")
    .whenNotMatchedInsert(values={
        "customer_id": "s.customer_id",
        "invoice_no": "s.invoice_no",
        "quantity": "s.quantity",
        "price": "s.price"
        # customer_type intentionally left out — merge only inserts what you tell it to
    })
    .execute()
)

In [89]:
(
    target_table.alias("t")
    .merge(merge_source_df.alias("s"), "t.customer_id = s.customer_id")
    .whenNotMatchedInsertAll()
    .execute()
)

### Why `whenNotMatchedInsertAll()` Didn't Add `customer_type`

- `INSERT *` in a `MERGE` is **not** a literal "copy every source column" — it's star-expansion resolved **against the target table's existing schema**. It only pulls source columns that already have a matching name in the target.
- Since `customer_type` has no corresponding column in the target table, it's simply **excluded** from the expansion — silently, with no error and no schema change.
- This makes `whenNotMatchedInsertAll()` behave identically to an explicit `whenNotMatchedInsert(values={...})` mapping in this scenario — both only insert columns the target already knows about.
- Schema evolution (`mergeSchema`/`autoMerge`) would be required to actually get `customer_type` added as a new column — without it, `MERGE` just quietly drops anything extra rather than failing like plain `.save()`/append does.

# Handling Schema Evolution in Spark Writes — Summary

- **New columns (top-level or nested):** add `.option("mergeSchema", "true")` on `.write` — works for plain appends and for new fields inside a `struct()` (use named fields via `.alias()` so Delta knows the key). New columns always land at the **end** of the schema, regardless of where they appear in the source.

- **Type widening (e.g., `INT` → `BIGINT`):** `mergeSchema=true` alone won't do it — you must first enable it on the table (`ALTER TABLE ... SET TBLPROPERTIES ('delta.enableTypeWidening'='true')`) **and** manually widen the specific column (`ALTER COLUMN ... TYPE BIGINT`) before writes with bigger values will succeed.

- **Column position:** no writer option handles this — it's always manual, via `ALTER TABLE ADD COLUMN col_type FIRST` or `AFTER other_col`. Even `MERGE`/`mergeSchema` writes that successfully add a new column will always append it at the end, never in a requested position.

- **`MERGE` vs plain append behavior:** plain `.write.append()` is strict — throws a schema mismatch on any unrecognized column unless `mergeSchema=true` is set. `MERGE`'s `whenNotMatchedInsertAll()` / `whenMatchedUpdateAll()` silently **drop** any source column not already in the target schema unless schema evolution is separately enabled — no error either way, just no column added.

- **Session vs writer-level control:** `.option("mergeSchema", "true")` works for `DataFrameWriter` writes (`.save()`, `.saveAsTable()`); for `MERGE` operations, use the session config instead: `spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")`.

Merge Schema Example:


In [96]:
merge_source_df.show()

In [97]:
merge_source_df.printSchema()

In [98]:
source_delta_table_df.printSchema()

In [99]:
source_delta_table_df.show(3)

In [100]:
merge_source_df.write.format('delta').mode('append').option('mergeSchema',  True).save(source_delta_table)

In [102]:
spark.read.format('delta').load(source_delta_table).show(3)

In [103]:
spark.read.format('delta').load(source_delta_table).filter(F.col("customer_id")>170).show(3)